# Evaluation Notebook

This notebook loads the trained cross-encoder and bi-encoder models, evaluates them on the development set, and combines them using a weighted ensemble.

It also saves the final prediction files that can be scored using the local scorer.

In [1]:
import os
import random
import json
from typing import Dict, List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    RobertaModel,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

Device: cuda
GPU: Tesla T4


In [ ]:
BASE_DIR = os.getcwd()

OUT_DIR = os.path.join(BASE_DIR, "outputs", "category_C")
OUTPUT_DIR = os.path.join(OUT_DIR, "cross_encoder")
BI_OUTPUT_DIR = os.path.join(OUT_DIR, "bi_encoder")
ENSEMBLE_OUTPUT_DIR = os.path.join(OUT_DIR, "ensemble")
SUBMISSION_DIR = os.path.join(OUT_DIR, "submissions")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BI_OUTPUT_DIR, exist_ok=True)
os.makedirs(ENSEMBLE_OUTPUT_DIR, exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

print("Output directory:", OUT_DIR)

Output directory: /content/outputs/category_C


In [ ]:
# paths and loading data

DEV_PATH = "data/dev.csv"

MODEL_NAME = "roberta-base"

assert os.path.exists(DEV_PATH), "dev.csv not found in data/ folder."

dev_df = pd.read_csv(DEV_PATH)
print("Dev shape:", dev_df.shape)
display(dev_df.head())

Dev shape: (6736, 3)


,premise,hypothesis,label
0,"By starting at the soft underbelly, the 16,000...","General Nelson A. Miles had 30,000 troops in h...",0
1,"The class had broken into a light sweat, but w...",The class grew more tense as time went on.,1
2,"Samson had his famous haircut here, but he wou...",It was unknown where exactly within the town S...,1
3,A man with a black shirt holds a baby while a ...,A darkly dressed man passes a crying baby to a...,0
4,I know that many of you are interested in addr...,The problems must be addressed,1


In [9]:
# =========================
# Data prep
# =========================
def guess_column(df: pd.DataFrame, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in cols_lower:
            return cols_lower[cand]
    return None

premise_col = guess_column(dev_df, ["premise", "sentence1", "text1", "s1"])
hypothesis_col = guess_column(dev_df, ["hypothesis", "sentence2", "text2", "s2"])
label_col = guess_column(dev_df, ["label", "gold_label", "class", "y"])

if premise_col is None or hypothesis_col is None or label_col is None:
    raise ValueError(f"Could not detect required columns. Found: {list(dev_df.columns)}")

dev_df[premise_col] = dev_df[premise_col].fillna("").astype(str).str.strip()
dev_df[hypothesis_col] = dev_df[hypothesis_col].fillna("").astype(str).str.strip()

dev_df = dev_df[(dev_df[premise_col] != "") & (dev_df[hypothesis_col] != "")].copy()

# Load cross-encoder metadata
with open(os.path.join(OUTPUT_DIR, "cross_encoder_metadata.json"), "r") as f:
    cross_metadata = json.load(f)

# Rebuild label mapping directly from dev labels
unique_labels = sorted(dev_df[label_col].unique().tolist())
label2id = {lab: i for i, lab in enumerate(unique_labels)}
id2label = {i: lab for lab, i in label2id.items()}

dev_df["label_id"] = dev_df[label_col].map(label2id)
if dev_df["label_id"].isna().sum() != 0:
    raise ValueError("Some labels in dev set could not be mapped.")

dev_df["label_id"] = dev_df["label_id"].astype(int)
num_labels = len(label2id)

# IMPORTANT: load tokenizer from tokenizer subfolder, not OUTPUT_DIR itself
tokenizer = AutoTokenizer.from_pretrained(
    os.path.join(OUTPUT_DIR, "cross_encoder_tokenizer")
)

MAX_LENGTH = cross_metadata["max_length"]
BATCH_SIZE = cross_metadata["batch_size"]
DROPOUT = cross_metadata["dropout"]
HIDDEN_DIM_1 = cross_metadata["hidden_dim_1"]
HIDDEN_DIM_2 = cross_metadata["hidden_dim_2"]

print("premise_col:", premise_col)
print("hypothesis_col:", hypothesis_col)
print("label_col:", label_col)
print("num_labels:", num_labels)
print("Tokenizer path:", os.path.join(OUTPUT_DIR, "cross_encoder_tokenizer"))

premise_col: premise
hypothesis_col: hypothesis
label_col: label
num_labels: 2
Tokenizer path: /content/outputs/category_C/cross_encoder/cross_encoder_tokenizer


In [10]:
# dataset classes

class NLIPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, premise_col: str, hypothesis_col: str, label_col: str, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.premise_col = premise_col
        self.hypothesis_col = hypothesis_col
        self.label_col = label_col
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        prem = row[self.premise_col]
        hyp = row[self.hypothesis_col]
        label = int(row[self.label_col])

        enc = self.tokenizer(
            prem,
            hyp,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )
        enc["labels"] = label
        return enc


class NLIBiEncoderDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, premise_col: str, hypothesis_col: str, label_col: str, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.premise_col = premise_col
        self.hypothesis_col = hypothesis_col
        self.label_col = label_col
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        prem = row[self.premise_col]
        hyp = row[self.hypothesis_col]
        label = int(row[self.label_col])

        prem_enc = self.tokenizer(
            prem,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )
        hyp_enc = self.tokenizer(
            hyp,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        return {
            "premise_input_ids": prem_enc["input_ids"],
            "premise_attention_mask": prem_enc["attention_mask"],
            "hypothesis_input_ids": hyp_enc["input_ids"],
            "hypothesis_attention_mask": hyp_enc["attention_mask"],
            "labels": label
        }


def bi_collate_fn(features):
    premise_features = [
        {"input_ids": f["premise_input_ids"], "attention_mask": f["premise_attention_mask"]}
        for f in features
    ]
    hypothesis_features = [
        {"input_ids": f["hypothesis_input_ids"], "attention_mask": f["hypothesis_attention_mask"]}
        for f in features
    ]

    premise_batch = tokenizer.pad(premise_features, padding=True, return_tensors="pt")
    hypothesis_batch = tokenizer.pad(hypothesis_features, padding=True, return_tensors="pt")
    labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

    return {
        "premise_input_ids": premise_batch["input_ids"],
        "premise_attention_mask": premise_batch["attention_mask"],
        "hypothesis_input_ids": hypothesis_batch["input_ids"],
        "hypothesis_attention_mask": hypothesis_batch["attention_mask"],
        "labels": labels
    }

In [11]:
dev_ds = NLIPairDataset(
    dev_df, tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

dev_loader = DataLoader(
    dev_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=data_collator
)

dev_bi_ds = NLIBiEncoderDataset(
    dev_df, tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)

dev_bi_loader = DataLoader(
    dev_bi_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=bi_collate_fn
)

## Model Definitions

The model classes are redefined here so that the saved weights can be loaded into the correct architectures for evaluation.

In [12]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class RobertaCustomNLI(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float = 0.2, hidden_dim_1: int = 256, hidden_dim_2: int = 128):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_dim_1),
            nn.LayerNorm(hidden_dim_1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.LayerNorm(hidden_dim_2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim_2, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_repr)


class RobertaBiEncoderNLI(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float = 0.2, hidden_dim_1: int = 256, hidden_dim_2: int = 128):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        combined_size = hidden_size * 3

        self.classifier = nn.Sequential(
            nn.Linear(combined_size, hidden_dim_1),
            nn.LayerNorm(hidden_dim_1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.LayerNorm(hidden_dim_2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim_2, num_labels)
        )

    def forward(
        self,
        premise_input_ids,
        premise_attention_mask,
        hypothesis_input_ids,
        hypothesis_attention_mask
    ):
        premise_outputs = self.encoder(input_ids=premise_input_ids, attention_mask=premise_attention_mask)
        hypothesis_outputs = self.encoder(input_ids=hypothesis_input_ids, attention_mask=hypothesis_attention_mask)

        u = mean_pooling(premise_outputs.last_hidden_state, premise_attention_mask)
        v = mean_pooling(hypothesis_outputs.last_hidden_state, hypothesis_attention_mask)

        features = torch.cat([u, v, torch.abs(u - v)], dim=-1)
        return self.classifier(features)

In [14]:
# load trained models

best_model = RobertaCustomNLI(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2
)
best_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "cross_encoder_model.pt"), map_location=device))
best_model.to(device)
best_model.eval()

best_bi_model = RobertaBiEncoderNLI(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2
)
best_bi_model.load_state_dict(torch.load(os.path.join(BI_OUTPUT_DIR, "bi_encoder_model.pt"), map_location=device))
best_bi_model.to(device)
best_bi_model.eval()

print("Both models loaded.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Both models loaded.


## Evaluation Functions

The helper functions in this section run inference on the development set and compute the main evaluation metrics for both models.

In [15]:
criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, dataloader, criterion):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(dataloader))
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1_macro, all_labels, all_preds


@torch.no_grad()
def evaluate_bi(model, dataloader, criterion):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Evaluating Bi-Encoder", leave=False):
        premise_input_ids = batch["premise_input_ids"].to(device)
        premise_attention_mask = batch["premise_attention_mask"].to(device)
        hypothesis_input_ids = batch["hypothesis_input_ids"].to(device)
        hypothesis_attention_mask = batch["hypothesis_attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            premise_input_ids=premise_input_ids,
            premise_attention_mask=premise_attention_mask,
            hypothesis_input_ids=hypothesis_input_ids,
            hypothesis_attention_mask=hypothesis_attention_mask
        )

        loss = criterion(logits, labels)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(dataloader))
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1_macro, all_labels, all_preds

In [16]:
# evaluate cross and bi models

dev_loss, dev_acc, dev_f1, dev_labels, dev_preds = evaluate(best_model, dev_loader, criterion)
bi_dev_loss, bi_dev_acc, bi_dev_f1, bi_dev_labels, bi_dev_preds = evaluate_bi(best_bi_model, dev_bi_loader, criterion)

print(f"Cross Dev Loss: {dev_loss:.4f}")
print(f"Cross Dev Acc: {dev_acc:.4f}")
print(f"Cross Dev Macro-F1: {dev_f1:.4f}")
print(classification_report(dev_labels, dev_preds))
print(confusion_matrix(dev_labels, dev_preds))

print(f"\nBi Dev Loss: {bi_dev_loss:.4f}")
print(f"Bi Dev Acc: {bi_dev_acc:.4f}")
print(f"Bi Dev Macro-F1: {bi_dev_f1:.4f}")
print(classification_report(bi_dev_labels, bi_dev_preds))
print(confusion_matrix(bi_dev_labels, bi_dev_preds))

Evaluating:   0%|          | 0/421 [00:00<?, ?it/s]

Evaluating Bi-Encoder:   0%|          | 0/421 [00:00<?, ?it/s]

Cross Dev Loss: 0.3609
Cross Dev Acc: 0.8806
Cross Dev Macro-F1: 0.8804
              precision    recall  f1-score   support

           0       0.88      0.87      0.88      3258
           1       0.88      0.89      0.89      3478

    accuracy                           0.88      6736
   macro avg       0.88      0.88      0.88      6736
weighted avg       0.88      0.88      0.88      6736

[[2828  430]
 [ 374 3104]]

Bi Dev Loss: 0.4748
Bi Dev Acc: 0.7904
Bi Dev Macro-F1: 0.7892
              precision    recall  f1-score   support

           0       0.81      0.74      0.77      3258
           1       0.78      0.84      0.80      3478

    accuracy                           0.79      6736
   macro avg       0.79      0.79      0.79      6736
weighted avg       0.79      0.79      0.79      6736

[[2415  843]
 [ 569 2909]]


In [17]:
# get logits for ensemble

@torch.no_grad()
def get_cross_logits(model, dataloader):
    model.eval()
    all_logits, all_labels = [], []

    for batch in tqdm(dataloader, desc="Cross logits", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    return torch.cat(all_logits, dim=0), torch.cat(all_labels, dim=0)


@torch.no_grad()
def get_bi_logits(model, dataloader):
    model.eval()
    all_logits, all_labels = [], []

    for batch in tqdm(dataloader, desc="Bi logits", leave=False):
        premise_input_ids = batch["premise_input_ids"].to(device)
        premise_attention_mask = batch["premise_attention_mask"].to(device)
        hypothesis_input_ids = batch["hypothesis_input_ids"].to(device)
        hypothesis_attention_mask = batch["hypothesis_attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            premise_input_ids=premise_input_ids,
            premise_attention_mask=premise_attention_mask,
            hypothesis_input_ids=hypothesis_input_ids,
            hypothesis_attention_mask=hypothesis_attention_mask
        )

        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    return torch.cat(all_logits, dim=0), torch.cat(all_labels, dim=0)


cross_logits_dev, cross_labels_dev = get_cross_logits(best_model, dev_loader)
bi_logits_dev, bi_labels_dev = get_bi_logits(best_bi_model, dev_bi_loader)

assert torch.equal(cross_labels_dev, bi_labels_dev), "Label order mismatch between cross and bi dev loaders."
dev_labels_np = cross_labels_dev.numpy()

Cross logits:   0%|          | 0/421 [00:00<?, ?it/s]

Bi logits:   0%|          | 0/421 [00:00<?, ?it/s]

In [18]:
# ensemble alpha search

def evaluate_weighted_ensemble(cross_logits, bi_logits, labels, alpha):
    final_logits = alpha * cross_logits + (1.0 - alpha) * bi_logits
    preds = torch.argmax(final_logits, dim=-1).numpy()
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    return acc, f1_macro, preds

alpha_results = []

for alpha in np.arange(0.0, 1.01, 0.1):
    acc, f1_macro, preds = evaluate_weighted_ensemble(
        cross_logits=cross_logits_dev,
        bi_logits=bi_logits_dev,
        labels=dev_labels_np,
        alpha=float(alpha)
    )
    alpha_results.append({
        "alpha": round(float(alpha), 2),
        "dev_acc": acc,
        "dev_macro_f1": f1_macro
    })

alpha_results_df = pd.DataFrame(alpha_results)
display(alpha_results_df.sort_values("dev_macro_f1", ascending=False))

best_alpha_row = alpha_results_df.sort_values("dev_macro_f1", ascending=False).iloc[0]
best_alpha = float(best_alpha_row["alpha"])

print("Best alpha:", best_alpha)
print("Best ensemble dev acc:", best_alpha_row["dev_acc"])
print("Best ensemble dev macro F1:", best_alpha_row["dev_macro_f1"])

,alpha,dev_acc,dev_macro_f1
8,0.8,0.883759,0.883553
7,0.7,0.883165,0.882964
6,0.6,0.882126,0.881919
9,0.9,0.881681,0.881474
10,1.0,0.880641,0.880441
5,0.5,0.880048,0.879834
4,0.4,0.874406,0.874146
3,0.3,0.859709,0.859305
2,0.2,0.838628,0.838049
1,0.1,0.816211,0.815373


Best alpha: 0.8
Best ensemble dev acc: 0.8837589073634204
Best ensemble dev macro F1: 0.8835533676971233


In [ ]:
ensemble_dev_acc, ensemble_dev_f1, ensemble_dev_preds = evaluate_weighted_ensemble(
    cross_logits=cross_logits_dev,
    bi_logits=bi_logits_dev,
    labels=dev_labels_np,
    alpha=best_alpha
)

print(f"Final Ensemble Dev Acc: {ensemble_dev_acc:.4f}")
print(f"Final Ensemble Dev Macro-F1: {ensemble_dev_f1:.4f}")
print(classification_report(dev_labels_np, ensemble_dev_preds))
print(confusion_matrix(dev_labels_np, ensemble_dev_preds))

ensemble_metadata = {
    "best_alpha": best_alpha,
    "dev_acc": ensemble_dev_acc,
    "dev_macro_f1": ensemble_dev_f1
}

with open(os.path.join(ENSEMBLE_OUTPUT_DIR, "ensemble_metadata.json"), "w") as f:
    json.dump(ensemble_metadata, f, indent=2)

np.savetxt(os.path.join(OUTPUT_DIR, "cross_encoder_predictions.txt"), dev_preds, fmt="%d")
np.savetxt(os.path.join(BI_OUTPUT_DIR, "bi_encoder_predictions.txt"), bi_dev_preds, fmt="%d")
np.savetxt(os.path.join(ENSEMBLE_OUTPUT_DIR, "ensemble_predictions.txt"), ensemble_dev_preds, fmt="%d")

print("Saved cross encoder predictions.")
print("Saved bi encoder predictions.")
print("Saved ensemble predictions.")

Final Ensemble Dev Acc: 0.8838
Final Ensemble Dev Macro-F1: 0.8836
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      3258
           1       0.88      0.90      0.89      3478

    accuracy                           0.88      6736
   macro avg       0.88      0.88      0.88      6736
weighted avg       0.88      0.88      0.88      6736

[[2835  423]
 [ 360 3118]]
Saved cross_encoder_predictions.txt
Saved bi_encoder_predictions.txt
Saved ensemble_predictions.txt
